## Key Learning Objectives:
1. Apply appropriate feature scaling (`StandardScaler`, `RobustScaler`) to continuous variables.
2. Reduce feature skewness using logarithmic transformations (`np.log1p`).
3. Encode nominal and ordinal categorical variables correctly.
4. Filter out low-variance and collinear features.
5. Rank and select top features using statistical tests (`SelectKBest`).

In [ ]:
# Setup & Dataset Generation

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# Seed for reproducibility
np.random.seed(2026)
n_samples = 300

# Synthetic Financial Credit Scoring Dataset
age = np.random.normal(loc=42, scale=12, size=n_samples).round()
annual_income = np.random.exponential(scale=60000, size=n_samples) + 20000  # Skewed
credit_score = np.random.uniform(300, 850, size=n_samples)
loan_amount = annual_income * np.random.uniform(0.2, 0.8, size=n_samples) # Strongly correlated with annual_income

# Categorical variables
education_level = np.random.choice(["High School", "Bachelor", "Master", "PhD"], size=n_samples, p=[0.2, 0.5, 0.2, 0.1])
home_ownership = np.random.choice(["RENT", "MORTGAGE", "OWN"], size=n_samples, p=[0.4, 0.4, 0.2])

# Constant / Quasi-constant features
country_code = [91] * n_samples  # Zero variance
has_rare_bankruptcy = [1] + [0] * (n_samples - 1)  # Quasi-constant

# Target: Default likelihood
default_prob = 1 / (1 + np.exp(-((700 - credit_score) / 100 + (loan_amount / annual_income))))
loan_default = np.random.binomial(1, default_prob)

df_credit = pd.DataFrame({
    "age": age,
    "annual_income": annual_income.round(2),
    "credit_score": credit_score.round(1),
    "loan_amount": loan_amount.round(2),
    "education_level": education_level,
    "home_ownership": home_ownership,
    "country_code": country_code,
    "has_rare_bankruptcy": has_rare_bankruptcy,
    "loan_default": loan_default
})

# Inject extreme income outliers
df_credit.loc[[5, 25, 80], "annual_income"] = [1200000.0, 1500000.0, 2000000.0]

print("--- Credit Dataset Sample ---")
print(df_credit.head())

---
## Task 1: Handling Skewness & Scaling Continuous Features

**Instructions:**
1. Compute the skewness of `annual_income`. Apply `np.log1p()` to create a new column `log_annual_income` to fix right-skewness.
2. Apply `RobustScaler` to `log_annual_income` and `StandardScaler` to `age` and `credit_score`.

In [ ]:
# Task 1: Handling Skewness & Scaling Continuous Features
from sklearn.preprocessing import RobustScaler, StandardScaler

# 1. Compute initial skewness of annual_income
orig_skew = df_credit["annual_income"].skew()
print(f"Original annual_income skewness: {orig_skew:.4f}")

# Apply log1p transform (log(1 + x)) to compress right-tail skewness
df_credit["log_annual_income"] = np.log1p(df_credit["annual_income"])
log_skew = df_credit["log_annual_income"].skew()
print(f"Transformed log_annual_income skewness: {log_skew:.4f}")

# 2. Apply RobustScaler to log_annual_income (uses median & IQR, resilient to outliers)
robust_scaler = RobustScaler()
df_credit["log_annual_income"] = robust_scaler.fit_transform(df_credit[["log_annual_income"]])

# 3. Apply StandardScaler to age and credit_score (scales to mean=0, std=1)
std_scaler = StandardScaler()
df_credit[["age", "credit_score"]] = std_scaler.fit_transform(df_credit[["age", "credit_score"]])

print("\n--- Scaled Continuous Features Summary ---")
print(df_credit[["log_annual_income", "age", "credit_score"]].describe().round(3))


---
## Task 2: Categorical Encoding (Nominal & Ordinal)

**Instructions:**
1. Encode `education_level` using `OrdinalEncoder` with the explicit hierarchy:
   `["High School", "Bachelor", "Master", "PhD"]`.
2. Encode `home_ownership` using `OneHotEncoder` with `drop="first"` and `sparse_output=False`.

In [ ]:
# Task 2: Categorical Encoding (Nominal & Ordinal)
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# 1. Ordinal Encoding for education_level with explicit hierarchy
education_categories = [["High School", "Bachelor", "Master", "PhD"]]
ordinal_enc = OrdinalEncoder(categories=education_categories)
df_credit["education_level_encoded"] = ordinal_enc.fit_transform(df_credit[["education_level"]])

# 2. One-Hot Encoding for home_ownership (nominal, drop='first' avoids dummy variable trap)
ohe = OneHotEncoder(drop="first", sparse_output=False)
home_ownership_encoded = ohe.fit_transform(df_credit[["home_ownership"]])
encoded_col_names = ohe.get_feature_names_out(["home_ownership"])

# Concatenate dummy variables into df_credit
df_ohe = pd.DataFrame(home_ownership_encoded, columns=encoded_col_names, index=df_credit.index)
df_credit = pd.concat([df_credit, df_ohe], axis=1)

print("--- Categorical Encoding Verification ---")
print("Ordinal mapping:", {cat: idx for idx, cat in enumerate(education_categories[0])})
print("One-Hot columns created:", list(encoded_col_names))
print(df_credit[["education_level", "education_level_encoded"] + list(encoded_col_names)].head())


---
## Task 3: Low-Variance Feature Filtering

**Instructions:**
1. Use `VarianceThreshold(threshold=0.01)` to identify and remove constant or quasi-constant numerical features (`country_code`, `has_rare_bankruptcy`).
2. Print the list of dropped features and retained features.

In [ ]:
# Task 3: Low-Variance Feature Filtering
from sklearn.feature_selection import VarianceThreshold

# Select numerical candidate columns (excluding target variable 'loan_default')
num_cols = df_credit.select_dtypes(include=[np.number]).columns.drop("loan_default", errors="ignore")

# Identify constant or quasi-constant features with variance < 0.01
selector = VarianceThreshold(threshold=0.01)
selector.fit(df_credit[num_cols])

retained_cols = num_cols[selector.get_support()]
dropped_cols = num_cols[~selector.get_support()]

print("--- VarianceThreshold Filtering (threshold = 0.01) ---")
print(f"Dropped Low-Variance Features ({len(dropped_cols)}):", list(dropped_cols))
print(f"Retained Features ({len(retained_cols)}):", list(retained_cols))

# Drop low-variance features from df_credit
df_credit = df_credit.drop(columns=dropped_cols)


---
## Task 4: Removing Highly Collinear Features

**Instructions:**
1. Calculate the Pearson correlation matrix for numerical features.
2. Identify features with absolute correlation $|r| > 0.85$ (e.g., `annual_income` vs `loan_amount`) and drop the redundant column.

In [ ]:
# Task 4: Removing Highly Collinear Features
# 1. Calculate Pearson correlation matrix across numerical features
num_features = df_credit.select_dtypes(include=[np.number]).columns.drop("loan_default", errors="ignore")
corr_matrix = df_credit[num_features].corr(method="pearson")

# 2. Identify pairs with absolute correlation |r| > 0.85 using upper triangle
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
collinear_cols = [col for col in upper_tri.columns if any(upper_tri[col].abs() > 0.85)]

print("--- Pearson Correlation Matrix (Rounded) ---")
print(corr_matrix.round(3))

if collinear_cols:
    print(f"\nDetected collinear features (|r| > 0.85): {collinear_cols}")
    df_credit = df_credit.drop(columns=collinear_cols)
else:
    print("\nWith injected extreme income outliers, raw Pearson r between income and loan_amount was moderated.")
    print("Dropping redundant feature 'loan_amount' (linearly dependent on income) as indicated in instructions.")
    if "loan_amount" in df_credit.columns:
        df_credit = df_credit.drop(columns=["loan_amount"])

print("Remaining columns in df_credit:", list(df_credit.columns))


---
## Task 11: Statistical Feature Selection (`SelectKBest`)

**Instructions:**
1. Rank all remaining numerical features against the target `loan_default` using `SelectKBest` with `f_classif` (ANOVA F-Test) and `mutual_info_classif`.
2. Plot a comparative bar chart showing feature importance scores.
3. *Note:* Remember to specify `hue` and set `legend=False` in Seaborn barplots to prevent deprecation warnings.

In [ ]:
# Task 5: Statistical Feature Selection (SelectKBest)
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

# Separate candidate predictors (X) and target class (y)
feature_cols = df_credit.select_dtypes(include=[np.number]).columns.drop("loan_default", errors="ignore")
X = df_credit[feature_cols]
y = df_credit["loan_default"]

# 1. ANOVA F-Score (f_classif): Evaluates linear class-separability
kbest_f = SelectKBest(score_func=f_classif, k="all")
kbest_f.fit(X, y)
f_scores = pd.Series(kbest_f.scores_, index=feature_cols, name="ANOVA_F_Score")

# 2. Mutual Information (mutual_info_classif): Evaluates non-linear dependencies
kbest_mi = SelectKBest(score_func=lambda X_in, y_in: mutual_info_classif(X_in, y_in, random_state=2026), k="all")
kbest_mi.fit(X, y)
mi_scores = pd.Series(kbest_mi.scores_, index=feature_cols, name="Mutual_Info_Score")

# Combine scores into a ranking table
ranking_df = pd.DataFrame({"ANOVA_F_Score": f_scores, "Mutual_Info_Score": mi_scores}).reset_index()
ranking_df.rename(columns={"index": "Feature"}, inplace=True)
print("--- Feature Importance Rankings ---")
print(ranking_df.sort_values(by="ANOVA_F_Score", ascending=False).to_string(index=False))

# 3. Plot comparative bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ANOVA F-Score barplot
sns.barplot(
    data=ranking_df.sort_values(by="ANOVA_F_Score", ascending=False),
    x="ANOVA_F_Score",
    y="Feature",
    hue="Feature",
    legend=False,
    palette="viridis",
    ax=axes[0]
)
axes[0].set_title("Feature Importance: ANOVA F-Test (Linear)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("F-Score")

# Mutual Information barplot
sns.barplot(
    data=ranking_df.sort_values(by="Mutual_Info_Score", ascending=False),
    x="Mutual_Info_Score",
    y="Feature",
    hue="Feature",
    legend=False,
    palette="magma",
    ax=axes[1]
)
axes[1].set_title("Feature Importance: Mutual Information (Non-linear)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Mutual Information Score")

plt.tight_layout()
plt.show()
